# ESR-rate spectrum versus a long driven Rabi pulse

This notebook compares two descriptions of the same isolated spin-1/2 transition:

- the stationary population-rate spectrum from `ESR.calc_FreqSweep_ESRrate`, and
- the final population after a finite coherent pulse from `CC.calc_RabiDetuningMap`.

The comparison is meaningful under the rotating-wave approximation (RWA), with a Markovian two-level bath, the same actual Rabi frequency, the same total $T_2$, and the same state-dependent current readout. We compare the **dark-subtracted final population/current**, not the fitted amplitude of transient Rabi oscillations and not a Ramsey spectrum.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
from qutip import Qobj
from spinfinity.helperfunctions import fit_fano

import spinfinity
from spinfinity import CC, ESR, SpinSys, bath, hamil, lind

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
print("Spinfinity source:", Path(spinfinity.__file__).resolve())

## 1. Static spin and tunnel-junction bath

Spinfinity uses meV for Hamiltonians, GHz for ordinary frequencies, ns for coherent evolution, and s$^{-1}$ for bath rates. The state indices below are energy ordered, and `Rates[channel, i, j]` means initial state $i$ and final state $j$.

The finite bias can pump the spin, so the RF-off stationary populations are calculated from the rate equation and later used as the initial density matrix.

In [ ]:
spin = SpinSys([0.5])
spin.gfactorvector = [2.0]
spin.B = [0.0, 0.0, 0.5]  # T
spin.BTip = [1e-9, 0.0, 0.0]  # T
spin.tip = "f"

spin.T = 0.5  # K
spin.V_DC = 10.0  # mV
spin.TipPolarization = [0.0, 0.0, 0.5]
spin.SamplePolarization = [0.0, 0.0, 0.0]
spin.G_st = 1e-10  # S
spin.G_ss = [1e-9]  # S
spin.G_tt = 1
spin.U = np.array([0.0])
spin.b0 = 0.0

hamil.calc_EigEnergies(spin, rebuildH=True)
bath.calc_Rates(spin)
bath.calc_Populations(spin, AllowPumping=True)

f_res = float(spin.E_All_inGHz[1] - spin.E_All_inGHz[0])
p_off = np.asarray(np.real_if_close(spin.Populations), dtype=float).copy()

print(f"Resonance frequency: {f_res:.9f} GHz")
print(f"RF-off stationary populations: {p_off}")
print(f"Population sum: {p_off.sum():.12f}")

## 2. Matched $T_1$, $T_2$, collapse operators, and readout

For a two-level population model,

$$
T_1=\frac{10^9}{\Gamma_{0\to1}+\Gamma_{1\to0}}\ \mathrm{ns},
\qquad
\frac{1}{T_2}=\frac{1}{2T_1}+\frac{1}{T_\phi}.
$$

The inelastic collapse operators use all positive rates without threshold filtering. Pure dephasing is added as $L_\phi=\sqrt{2/T_\phi}\,S_{z'}$. Elastic collapse operators are deliberately omitted: adding them would introduce extra dephasing not included in the $T_2$ passed to the ESR-rate calculation.

In [ ]:
gamma_up = float(spin.Rates_Summed[0, 1])  # s^-1
gamma_down = float(spin.Rates_Summed[1, 0])  # s^-1
T1 = 1e9 / (gamma_up + gamma_down)  # ns
T_phi = 200.0  # ns
T2 = 1.0 / (1.0 / (2.0 * T1) + 1.0 / T_phi)  # ns

# RotFrameSecular creates field-aligned Sx_rot, Sy_rot, and Sz_rot.
CC.RotFrameSecular(spin, f_res)
inelastic_collapse_ops = lind.calc_collapseOperators(
    spin, threshold_percent=None, recalcRates=False
)
pure_dephasing_op = np.sqrt(2.0 / T_phi) * Qobj(spin.Sz_rot)
spin.collapse_ops = [*inelastic_collapse_ops, pure_dephasing_op]

# RF-off stationary density matrix in the product-spin basis.
rho_off = spin.eigVectors @ np.diag(p_off) @ spin.eigVectors.conj().T
spin.psi0 = Qobj(rho_off)

# Exact excited-state projector and state-dependent current coefficients.
ket_excited = Qobj(spin.eigVectors[:, 1])
projector_excited = ket_excited * ket_excited.dag()
state_currents = np.real(const.e * np.sum(spin.Rates[0] - spin.Rates[1], axis=1))
current_contrast = state_currents[1] - state_currents[0]

tau_scale = max(T1, T2)
tau_long = 20.0 * tau_scale

print(f"Upward/downward rates: {gamma_up:.6e} / {gamma_down:.6e} s^-1")
print(f"T1: {T1:.7f} ns")
print(f"T_phi: {T_phi:.1f} ns")
print(f"Total T2: {T2:.7f} ns")
print(f"Inelastic collapse operators: {len(inelastic_collapse_ops)}")
print(f"Conservative final pulse: {tau_long:.3f} ns = 20 max(T1, T2)")
print(f"State-current contrast I1-I0: {current_contrast * 1e15:.6f} fA")

## 3. Match `B1` to `RabiRate`

The coherent solver uses $\Omega=2\pi\,\texttt{RabiRate}$ in rad/ns. The ESR-rate implementation uses

$$
\Omega=\gamma B_1\left|\langle 0|S_x|1\rangle\right|,
\qquad
\gamma=\frac{|g|\mu_B}{\hbar}10^{-9}.
$$

Therefore `B1` must be derived from the requested `RabiRate`. For this field-aligned spin-1/2, $|\langle0|S_x|1\rangle|=1/2$ and a 10 MHz Rabi rate requires about 0.7145 mT peak linear drive.

In [ ]:
gamma = abs(spin.gfactorvector[spin.ReadoutSpin]) * spin.muBHz / const.hbar * 1e-9
Sx_energy = spin.eigVectors.conj().T @ spin.Sx[spin.ReadoutSpin] @ spin.eigVectors
matrix_element = abs(Sx_energy[0, 1])


def b1_for_rabi_rate(rabi_rate):
    """Return the peak linear B1 field in tesla for a Rabi rate in GHz.

    Parameters
    ----------
    rabi_rate : float
        Ordinary Rabi frequency in GHz.

    Returns
    -------
    float
        Peak amplitude of the linearly oscillating field in tesla.
    """
    omega = 2.0 * np.pi * rabi_rate
    return omega / (gamma * matrix_element)


rabi_rates = np.array([1.0, 10.0]) * 1e-3  # GHz
for rabi_rate in rabi_rates:
    print(
        f"{rabi_rate * 1e3:4.1f} MHz -> B1 = {b1_for_rabi_rate(rabi_rate) * 1e3:.8f} mT"
    )

## 4. Transient detuning map at 10 MHz

The map includes zero time and extends to $20\max(T_1,T_2)$. Its measured quantity is the excited-state population, converted to the same signed, dark-subtracted current used by the rate model:

$$
\Delta I(t,f)=(I_1-I_0)\,[p_1(t,f)-p_1^{\rm off}].
$$

The early-time fringes are coherent Rabi dynamics. Only the converged endpoint should match the continuous-wave rate spectrum.

In [ ]:
detuning = np.linspace(-40.0, 40.0, 121) * 1e-3  # GHz
frequency_array = f_res + detuning
requested_times = np.array([0.5 * T2, 2.0 * T2, 5.0 * T2, 12.0 * tau_scale])
pulse_times = np.unique(
    np.concatenate((np.linspace(0.0, tau_long, 241), requested_times, [tau_long]))
)

rabi_map_rate = 10e-3  # GHz
spin.RabiRate = rabi_map_rate
solver_options = {"method": "diag", "store_states": False, "progress_bar": ""}

time_map, frequencies_map, p1_map = CC.calc_RabiDetuningMap(
    spin,
    pulse_times,
    frequency_array,
    e_ops=projector_excited,
    solver_options=solver_options,
    n_jobs=1,
)
p1_map = np.asarray(np.real_if_close(p1_map), dtype=float)
current_map = current_contrast * (p1_map - p_off[1])

B1_map = b1_for_rabi_rate(rabi_map_rate)
_, current_rate_map, I_off_map = ESR.calc_FreqSweep_ESRrate(
    spin,
    freq_array=frequency_array,
    B1=B1_map,
    T2=T2,
    norm=False,
)

limit_fa = np.max(np.abs(current_map)) * 1e15
fig, ax = plt.subplots(figsize=(8.5, 4.5), constrained_layout=True)
image = ax.pcolormesh(
    time_map,
    detuning * 1e3,
    current_map * 1e15,
    shading="auto",
    cmap="RdBu_r",
    vmin=-limit_fa,
    vmax=limit_fa,
)
ax.set(
    xlabel="Pulse duration (ns)",
    ylabel="Detuning (MHz)",
    title="Transient current response: RabiRate = 10 MHz",
)
fig.colorbar(image, ax=ax, label=r"$\Delta I(t,f)$ (fA)")
plt.show()

### Frequency cuts before and after convergence

The curves are divided by the **single ESR-rate peak magnitude**, preserving their sign and relative amplitude. They are not independently normalized.

In [ ]:
cut_targets = np.array([0.5 * T2, 2.0 * T2, tau_long])
cut_indices = [int(np.argmin(abs(time_map - target))) for target in cut_targets]
peak_rate_map = np.max(np.abs(current_rate_map))

fig, ax = plt.subplots(figsize=(8.5, 4.5), constrained_layout=True)
for index in cut_indices:
    ax.plot(
        detuning * 1e3,
        current_map[:, index] / peak_rate_map,
        label=f"Pulse endpoint: {time_map[index]:.1f} ns",
    )
ax.plot(
    detuning * 1e3,
    current_rate_map / peak_rate_map,
    "k--",
    linewidth=2,
    label="Stationary ESR-rate spectrum",
)
ax.set(
    xlabel="Detuning (MHz)",
    ylabel=r"Signed $\Delta I$ / ESR peak magnitude",
    title="Transient cuts approach the stationary spectrum",
)
ax.legend()
plt.show()

## 5. Analytical stationary line shape

The two-level Bloch steady state predicts

$$
\Delta I(f)=\frac{(I_1-I_0)(p_0^{\rm off}-p_1^{\rm off})}{2}
\frac{\Omega^2T_1T_2}
{1+[2\pi(f-f_0)T_2]^2+\Omega^2T_1T_2},
$$

with

$$
\mathrm{FWHM}=\frac{\sqrt{1+\Omega^2T_1T_2}}{\pi T_2}.
$$

The helper named `fit_fano` fits the symmetric Lorentzian form used here. We compare its signed amplitude, center, and FWHM for both the rate spectrum and the long-pulse current.

In [ ]:
def stationary_current(detuning_values, rabi_rate):
    """Return the analytical dark-subtracted current in amperes.

    Parameters
    ----------
    detuning_values : array_like
        Drive detunings from resonance in GHz.
    rabi_rate : float
        Ordinary Rabi frequency in GHz.

    Returns
    -------
    numpy.ndarray
        Stationary current contrast in amperes.
    """
    omega = 2.0 * np.pi * rabi_rate
    saturation = omega**2 * T1 * T2
    prefactor = current_contrast * (p_off[0] - p_off[1]) / 2.0
    denominator = 1.0 + (2.0 * np.pi * detuning_values * T2) ** 2 + saturation
    return prefactor * saturation / denominator


def theoretical_fwhm(rabi_rate):
    """Return the power-broadened FWHM in GHz.

    Parameters
    ----------
    rabi_rate : float
        Ordinary Rabi frequency in GHz.

    Returns
    -------
    float
        Full width at half maximum in GHz.
    """
    omega = 2.0 * np.pi * rabi_rate
    return np.sqrt(1.0 + omega**2 * T1 * T2) / (np.pi * T2)


analytic_map = stationary_current(detuning, rabi_map_rate)
rate_analytic_error = np.max(np.abs(current_rate_map - analytic_map)) / peak_rate_map

print(f"10 MHz theoretical FWHM: {theoretical_fwhm(rabi_map_rate) * 1e3:.7f} MHz")
print(
    f"Rate-model vs analytical maximum relative discrepancy: {rate_analytic_error:.3e}"
)

## 6. Long-pulse comparison at two drive strengths

The 10 MHz endpoint is reused from the full transient map. For 1 MHz, only the initial and final times are requested because the purpose is the stationary comparison. The same `spin`, bath, collapse operators, frequency grid, and current observable are reused throughout.

In [ ]:
comparisons = {}

for rabi_rate in rabi_rates:
    spin.RabiRate = float(rabi_rate)
    B1 = b1_for_rabi_rate(rabi_rate)
    _, current_rate, I_off = ESR.calc_FreqSweep_ESRrate(
        spin,
        freq_array=frequency_array,
        B1=B1,
        T2=T2,
        norm=False,
    )

    if np.isclose(rabi_rate, rabi_map_rate):
        current_long = current_map[:, -1]
    else:
        _, _, p1_endpoint = CC.calc_RabiDetuningMap(
            spin,
            np.array([0.0, tau_long]),
            frequency_array,
            e_ops=projector_excited,
            solver_options=solver_options,
            n_jobs=1,
        )
        p1_endpoint = np.asarray(np.real_if_close(p1_endpoint), dtype=float)
        current_long = current_contrast * (p1_endpoint[:, -1] - p_off[1])

    current_theory = stationary_current(detuning, rabi_rate)
    peak = np.max(np.abs(current_rate))
    relative_error = np.max(np.abs(current_long - current_rate)) / peak

    rate_fit = fit_fano(frequency_array, current_rate)
    long_fit = fit_fano(frequency_array, current_long)
    comparisons[float(rabi_rate)] = {
        "B1": B1,
        "rate": current_rate,
        "long": current_long,
        "theory": current_theory,
        "relative_error": relative_error,
        "rate_fit": rate_fit,
        "long_fit": long_fit,
        "I_off": I_off,
    }

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, rabi_rate in zip(axes, rabi_rates, strict=True):
    item = comparisons[float(rabi_rate)]
    ax.plot(detuning * 1e3, item["rate"] * 1e15, "-", label="ESR-rate")
    ax.plot(detuning * 1e3, item["long"] * 1e15, "o", ms=3, label="Long pulse")
    ax.plot(detuning * 1e3, item["theory"] * 1e15, "k--", label="Bloch stationary")
    ax.set(
        xlabel="Detuning (MHz)",
        ylabel=r"$\Delta I$ (fA)",
        title=f"RabiRate = {rabi_rate * 1e3:.0f} MHz",
    )
    ax.legend(fontsize=8)
plt.show()

In [ ]:
header = (
    "Rabi   B1       FWHM theory/rate/long       "
    "Amplitude theory/rate/long       max |long-rate|/peak"
)
print(header)
print("(MHz)  (mT)              (MHz)                         (fA)")

for rabi_rate in rabi_rates:
    item = comparisons[float(rabi_rate)]
    rate_parameters = item["rate_fit"][2]
    long_parameters = item["long_fit"][2]
    theory_amplitude = stationary_current(np.array([0.0]), rabi_rate)[0]
    print(
        f"{rabi_rate * 1e3:5.1f}  {item['B1'] * 1e3:7.5f}   "
        f"{theoretical_fwhm(rabi_rate) * 1e3:9.5f} / "
        f"{rate_parameters[2] * 1e3:9.5f} / "
        f"{long_parameters[2] * 1e3:9.5f}     "
        f"{theory_amplitude * 1e15:9.5f} / "
        f"{rate_parameters[0] * 1e15:9.5f} / "
        f"{long_parameters[0] * 1e15:9.5f}     "
        f"{item['relative_error']:.3e}"
    )

assert np.isclose(T1, 38.0512271, rtol=2e-7)
assert np.isclose(T2, 55.1262425, rtol=2e-7)
assert np.isclose(b1_for_rabi_rate(10e-3), 0.71447735e-3, rtol=2e-7)
assert np.isclose(theoretical_fwhm(10e-3), 17.5910155e-3, rtol=2e-7)
assert max(item["relative_error"] for item in comparisons.values()) < 1e-5
print("\nAll numerical consistency assertions passed.")

### Convergence is controlled by the driven dynamics

$\tau\gg T_2$ alone is not a general long-pulse criterion when $T_1$ is longer. Here the final duration is tied to the larger of $T_1$ and $T_2$. The next check shows how the 10 MHz spectrum approaches its stationary limit at several actual endpoints.

In [ ]:
convergence_targets = np.array(
    [0.5 * T2, 2.0 * T2, 5.0 * T2, 12.0 * tau_scale, tau_long]
)
convergence_errors = []

for target in convergence_targets:
    index = int(np.argmin(abs(time_map - target)))
    error = np.max(np.abs(current_map[:, index] - current_rate_map)) / peak_rate_map
    convergence_errors.append(error)
    print(f"t = {time_map[index]:9.3f} ns: max discrepancy / ESR peak = {error:.3e}")

fig, ax = plt.subplots(figsize=(6.5, 4), constrained_layout=True)
ax.semilogy(convergence_targets, convergence_errors, "o-")
ax.axvline(T1, color="tab:orange", linestyle=":", label=r"$T_1$")
ax.axvline(T2, color="tab:green", linestyle=":", label=r"$T_2$")
ax.set(
    xlabel="Pulse endpoint (ns)",
    ylabel="Maximum discrepancy / ESR peak",
    title="Convergence toward the stationary 10 MHz spectrum",
)
ax.legend()
plt.show()

## Conclusion

With matched bath rates, total $T_2$, drive matrix element, and population-based current detection, the long-pulse Lindblad result converges to the ESR-rate spectrum in signed, unnormalized current. Increasing the drive from 1 to 10 MHz increases the amplitude and broadens the line, as expected from saturation and power broadening.

The agreement does **not** mean that transient Rabi oscillations and a continuous-wave spectrum are identical. It can fail for a nonconverged pulse, mismatched $T_2$, different readout observables, multilevel coherence, non-Markovian noise, or an added homodyne channel.

### References

- [Ithier et al., *Phys. Rev. B* 72, 134519 (2005)](https://pure.royalholloway.ac.uk/ws/files/4707315/PhysRevB.72.134519.pdf), Eqs. (11)-(12): $T_2^{-1}=\tfrac12T_1^{-1}+T_\phi^{-1}$ and $T_1^{-1}$ is the sum of excitation and relaxation rates; the paper also states the short-correlated, weak-noise boundary of the Bloch-Redfield treatment.
- [Nicot, Fleury, and Leblond, *C. R. Chimie* 11, 506-514 (2008)](https://comptes-rendus.academie-sciences.fr/chimie/articles/10.1016/j.crci.2007.11.004/), Eq. (2): rotating-frame Bloch dynamics with $T_1$ and $T_2$ during an RF pulse.
- [Willke et al., *Science Advances* 4, eaaq1543 (2018)](https://doi.org/10.1126/sciadv.aaq1543): ESR-STM saturation and power broadening.
- [Seifert et al., supplementary Section S2 (2020)](https://arxiv.org/pdf/2005.07455): population-based dc current versus homodyne detection, and the co-rotating half of a linearly polarized field.

The explicit current mapping and numerical equality tested here are model-specific deductions in Spinfinity's conventions, not formulas quoted verbatim from those papers.